Statistical Testing



In [2]:
import pandas as pd

In [7]:
df = pd.read_csv('/tmp/ab_test/marketing_AB.csv')

In [8]:
df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [9]:
print(df['test group'].value_counts(dropna=False))

test group
ad     564577
psa     23524
Name: count, dtype: int64


In [10]:
print(df['converted'].value_counts(dropna=False))

converted
False    573258
True      14843
Name: count, dtype: int64


In [11]:
df.dtypes

Unnamed: 0       int64
user id          int64
test group         str
converted         bool
total ads        int64
most ads day       str
most ads hour    int64
dtype: object

In [12]:
ab_test_summary = df.groupby("test group").agg(
    users = ('user id','nunique'),
    converted = ('converted','sum'),
    avg_ads_seen = ('total ads','mean'),
    conversion_rate = ('converted','mean')
)
ab_test_summary['conversion_rate'] = ab_test_summary['conversion_rate']*100
ab_test_summary

,users,converted,avg_ads_seen,conversion_rate
test group,,,,
ad,564577,14423,24.823365,2.554656
psa,23524,420,24.761138,1.785411


In [23]:
ad_rate = ab_test_summary.loc['ad','conversion_rate']
psa_rate = ab_test_summary.loc['psa','conversion_rate']

absolute_lift = ad_rate - psa_rate
relative_lift = (absolute_lift/psa_rate)*100
print(f"Absolute lift : {round(absolute_lift,4)} percentage points")
print(f"Relaive lift : {round(relative_lift,2)}%")

Absolute lift : 0.7692 percentage points
Relaive lift : 43.09%


Two proportion Z-test



In [25]:
from statsmodels.stats.proportion import (proportions_ztest,confint_proportions_2indep)

In [40]:
# Sample sizes 
n_ad = int(ab_test_summary.loc['ad','users'])
n_psa = int(ab_test_summary.loc['psa','users'])



# number of conversions
conv_ad = int(ab_test_summary.loc['ad','converted'])
conv_psa = int(ab_test_summary.loc['psa','converted'])



# Two proportion z-test
z_stat,p_value = proportions_ztest(
    count = [conv_ad,conv_psa],
    nobs =[n_ad,n_psa],
    alternative='two-sided'
)


# 95% confidence interval for the differnce in conversion rate 
ci_low,ci_high = confint_proportions_2indep(
    count1 = conv_ad,
    nobs1= n_ad,
    count2 = conv_psa,
    nobs2= n_psa,
    method='newcomb',
    compare='diff',
    alpha = 0.05
)


print(f"Z Statistic : {round(z_stat,4)}")
print(f"P Value : {p_value}")
print(f"95% CI for absolute lift (percentage points) :   {round(ci_low*100,4)}, {round(ci_high*100,4)} ") 

Z Statistic : 7.3701
P Value : 1.7052807161559727e-13
95% CI for absolute lift (percentage points) :   0.5873, 0.936 


In [42]:
# Summary 
print(f"Ad group conversion rate : {conv_ad}")
print(f"PSA group conversion rate : {conv_psa}")
print(f"Absolute lift : {absolute_lift} ")
print(f"Relative lift : {relative_lift} ")
print(f"Z Statistic : {round(z_stat,4)}")
print(f"P Value : {p_value}")
print(f"95% CI for absolute lift (percentage points) :   {round(ci_low*100,4)}, {round(ci_high*100,4)} ") 

Ad group conversion rate : 14423
PSA group conversion rate : 420
Absolute lift : 0.7692453192201518 
Relative lift : 43.085064022225836 
Z Statistic : 7.3701
P Value : 1.7052807161559727e-13
95% CI for absolute lift (percentage points) :   0.5873, 0.936 


In [44]:
# Additional conversion per 100 k new users
ad_rate = ab_test_summary.loc['ad', 'conversion_rate']
psa_rate = ab_test_summary.loc['psa', 'conversion_rate']

extra_conversions_per_100k = (
    (ad_rate - psa_rate) / 100
) * 100_000

print(round(extra_conversions_per_100k))

769


In [66]:
# daily conversion analysis 

day_analysis = df.groupby(['most ads day','test group']).agg(
    users = ('user id','nunique'),
    converted = ('converted','sum'),
    conversion_rate = ('converted','mean')
)
day_analysis['conversion_rate']=day_analysis['conversion_rate']*100
day_analysis=day_analysis.sort_values(['most ads day','test group'])
day_analysis

users  converted  conversion_rate
most ads day test group                                   
Friday       ad          88805       1995         2.246495
             psa          3803         62         1.630292
Monday       ad          83571       2778         3.324120
             psa          3502         79         2.255854
Saturday     ad          78802       1679         2.130657
             psa          2858         40         1.399580
Sunday       ad          82332       2027         2.461983
             psa          3059         63         2.059497
Thursday     ad          79077       1711         2.163714
             psa          3905         79         2.023047
Tuesday      ad          74572       2270         3.044038
             psa          2907         42         1.444788
Wednesday    ad          77418       1963         2.535586
             psa          3490         55         1.575931

In [71]:
# hourly conversion analysis
hour_analysis= df.groupby(['most ads hour','test group']).agg(
    users = ('user id','nunique'),
    converted = ('converted','sum'),
    conversion_rate = ('converted','mean')
)
hour_analysis['conversion_rate'] = hour_analysis['conversion_rate']*100
hour_analysis = hour_analysis.sort_values(['most ads hour','test group'])
hour_analysis

users  converted  conversion_rate
most ads hour test group                                   
0             ad           5309        102         1.921266
              psa           227          0         0.000000
1             ad           4615         62         1.343445
              psa           187          0         0.000000
2             ad           5152         39         0.756988
              psa           181          0         0.000000
3             ad           2590         27         1.042471
              psa            89          1         1.123596
4             ad            694         11         1.585014
              psa            28          0         0.000000
5             ad            742         16         2.156334
              psa            23          0         0.000000
6             ad           1985         46         2.317380
              psa            83          0         0.000000
7             ad           6168        114         1.848249
              psa           237          2         0.843882
8             ad          16968        337         1.986091
              psa           659          7         1.062215
9             ad          29802        582         1.952889
              psa          1202         13         1.081531
10            ad          37454        818         2.184012
              psa          1485         20         1.346801
11            ad          44149        992         2.246937
              psa          2061         30         1.455604
12            ad          45238       1092         2.413900
              psa          2060         35         1.699029
13            ad          45485       1140         2.506321
              psa          2170         36         1.658986
14            ad          43779       1251         2.857534
              psa          1869         30         1.605136
15            ad          42855       1279         2.984483
              psa          1828         46         2.516411
16            ad          35963       1111         3.089286
              psa          1604         45         2.805486
17            ad          33605        959         2.853742
              psa          1383         28         2.024584
18            ad          31052        853         2.747005
              psa          1271         32         2.517703
19            ad          29169        782         2.680928
              psa          1183         29         2.451395
20            ad          27846        843         3.027365
              psa          1077         19         1.764160
21            ad          28895        843         2.917460
              psa          1081         24         2.220167
22            ad          25515        675         2.645503
              psa           917         15         1.635769
23            ad          19547        449         2.297028
              psa           619          8         1.292407

In [72]:
control_rate = 1.785411 / 100
treatment_rate = 2.554656 / 100

users = 10_000

control_conversions = users * control_rate
treatment_conversions = users * treatment_rate

additional_conversions = treatment_conversions - control_conversions

print("Control conversions:", control_conversions)
print("Treatment conversions:", treatment_conversions)
print("Additional conversions:", additional_conversions)

Control conversions: 178.5411
Treatment conversions: 255.4656
Additional conversions: 76.9245
